# L20: LLM应用开发实战


# L20: LLM应用开发实战

**课时**: 2 课时（90 分钟/课时）

## 1. 学习目标

完成本课程后，学员能够：
1. 使用LangChain构建完整的LLM应用（聊天机器人、文档问答、数据分析助手）
2. 掌握LangChain的核心组件：Chain、Prompt Template、Output Parser、Memory
3. 实现流式输出（Streaming）和异步处理，提升用户体验
4. 部署和监控LLM应用，处理API限流、错误重试等生产环境问题
5. 完成一个完整的LLM应用项目：从需求到上线

## 2. 核心知识点

### 2.1 LangChain核心概念

**Prompt Template**: 可复用的提示词模板，支持变量替换


In [ ]:
from langchain.prompts import PromptTemplate

# 基本模板
template = PromptTemplate(
    input_variables=["product", "audience"],
    template="为{audience}写一个关于{product}的广告文案，要求吸引人、有说服力。"
)

prompt = template.format(product="新能源汽车", audience="年轻上班族")
print(prompt)
# 输出：为年轻上班族写一个关于新能源汽车的广告文案，要求吸引人、有说服力。


**Chain**: 将多个组件串联成管道


In [ ]:
from langchain.chains import LLMChain

chain = LLMChain(llm=llm, prompt=template)
result = chain.run(product="智能手机", audience="大学生")


**Memory**: 对话历史管理


In [ ]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 在Chain中使用
chain = LLMChain(llm=llm, prompt=prompt, memory=memory)


### 2.2 常用Chain类型

**SimpleSequentialChain**: 顺序执行，上一个输出作为下一个输入


In [ ]:
from langchain.chains import SimpleSequentialChain

# Step 1: 提取关键词
prompt1 = PromptTemplate(
    input_variables=["text"],
    template="从以下文本中提取5个关键词：{text}"
)
chain1 = LLMChain(llm=llm, prompt=prompt1, output_key="keywords")

# Step 2: 生成描述
prompt2 = PromptTemplate(
    input_variables=["keywords"],
    template="为以下关键词生成一句话描述：{keywords}"
)
chain2 = LLMChain(llm=llm, prompt=prompt2, output_key="description")

# 串联
overall_chain = SimpleSequentialChain(chains=[chain1, chain2], verbose=True)
result = overall_chain.run("人工智能是当前最热门的技术领域之一")
print(result)


**RouterChain**: 根据输入动态选择处理Chain


In [ ]:
from langchain.chains import RouterChain
from langchain.chains.llm_math.chain import LLMMathChain
from langchain.chains.conversational_retrieval.prompt import QA_PROMPT

# 定义多个子Chain
math_chain = LLMMathChain(llm=llm)
qa_chain = ...  # 问答Chain

# 路由选择Prompt
router_template = """
根据用户输入选择一个处理方式：

输入：{input}

选项：
1. 数学计算类 → 回答"数学"
2. 知识问答类 → 回答"问答"
3. 其他 → 回答"通用"

你的选择：
"""

router_chain = LLMChain(llm=llm, prompt=PromptTemplate(...))


### 2.3 构建文档问答系统


In [ ]:
"""
L20-doc-qa.py
使用LangChain构建文档问答系统
"""

from langchain.document_loaders import TextLoader, PDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

# ============ 1. 加载文档 ============
# loader = PDFLoader("document.pdf")
# documents = loader.load()

# 示例文档
documents = [
    "Transformer是由Google在2017年提出的革命性架构。",
    "它使用了自注意力机制（Self-Attention）来替代传统的RNN。",
    "BERT和GPT都是基于Transformer的重要模型。",
    "ChatGPT是基于GPT架构的对话应用，通过RLHF训练。"
]

# ============ 2. 分块 ============
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = text_splitter.split_text(" ".join(documents))

# ============ 3. 向量化 ============
# embeddings = OpenAIEmbeddings()
# vectorstore = Chroma.from_texts(chunks, embeddings, persist_directory="./db")

# ============ 4. 构建QA Chain ============
# llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
# qa_chain = RetrievalQA.from_chain_type(
#     llm=llm,
#     chain_type="stuff",  # 将所有检索内容塞入提示词
#     retriever=vectorstore.as_retriever(),
#     return_source_documents=True  # 返回源文档用于验证
# )

# ============ 5. 查询 ============
# question = "Transformer是什么？"
# result = qa_chain({"query": question})
# print(f"答案：{result['result']}")
# print(f"来源：{result['source_documents']}")


### 2.4 流式输出与异步处理

**流式输出 (Streaming)**:


In [ ]:
from langchain.callbacks import streaming_stdout

# 启用流式输出
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    streaming=True,  # 启用流式
    callbacks=[streaming_stdout.CallbackHandler()]
)

# 生成
for token in llm.stream("写一首关于春天的诗"):
    print(token, end="", flush=True)


**异步处理**:


In [ ]:
import asyncio
from langchain.chat_models import ChatOpenAI

async def async_generate():
    llm = ChatOpenAI(model="gpt-3.5-turbo")
    
    # 并发生成多个任务
    tasks = [
        llm.agenerate(["解释量子计算"]) for _ in range(3)
    ]
    
    results = await asyncio.gather(*tasks)
    return results

# 运行
results = asyncio.run(async_generate())


### 2.5 生产环境部署

**错误处理与重试**:


In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", max_retries=3)

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def call_llm_with_retry(prompt):
    try:
        response = llm.predict(prompt)
        return response
    except Exception as e:
        print(f"API调用失败: {e}")
        raise


**API限流处理**:


In [ ]:
import time
from langchain.chat_models import ChatOpenAI

class RateLimitedLLM:
    def __init__(self, calls_per_minute=60):
        self.calls_per_minute = calls_per_minute
        self.calls = []
        
    def generate(self, prompt):
        # 检查限流
        now = time.time()
        self.calls = [t for t in self.calls if now - t < 60]  # 保留最近60秒的调用
        
        if len(self.calls) >= self.calls_per_minute:
            sleep_time = 60 - (now - self.calls[0])
            print(f"限流：等待 {sleep_time:.1f} 秒")
            time.sleep(sleep_time)
        
        self.calls.append(time.time())
        return self.llm.predict(prompt)


## 3. 代码示例


In [ ]:
"""
L20-full-app.py
完整LLM应用：从需求到部署
"""

from langchain.agents import AgentType, initialize_agent
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.callbacks import get_openai_callback
import streamlit as st

# ============ 应用：AI数据分析助手 ============
"""
使用Streamlit构建Web界面：
1. 用户上传CSV/Excel文件
2. 用自然语言提问关于数据的问题
3. LLM解析问题 → 生成代码 → 执行 → 返回结果
"""

class DataAnalysisApp:
    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
        self.memory = ConversationBufferMemory(memory_key="chat_history")
        
        # 定义工具
        self.tools = [...]  # 搜索、代码执行、文件读写
        
        # 构建Agent
        self.agent = initialize_agent(
            self.tools,
            self.llm,
            agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
            memory=self.memory,
            verbose=True
        )
    
    def chat(self, user_input):
        """处理用户输入"""
        with get_openai_callback() as cb:
            response = self.agent.run(user_input)
            
        print(f"Token使用: {cb.total_tokens}")
        return response

# ============ Streamlit Web应用 ============
"""
import streamlit as st

st.set_page_config(page_title="AI数据分析助手", page_icon="🤖")

st.title("🤖 AI数据分析助手")
st.markdown("上传数据文件，用自然语言提问，我会帮你分析！")

# 文件上传
uploaded_file = st.file_uploader("上传CSV或Excel文件", type=["csv", "xlsx"])

if uploaded_file:
    # 加载数据
    df = load_data(uploaded_file)
    st.success(f"已加载 {len(df)} 行数据")
    
    # 显示数据预览
    st.dataframe(df.head())
    
    # 对话界面
    if "messages" not in st.session_state:
        st.session_state.messages = []
    
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])
    
    # 用户输入
    if prompt := st.chat_input("请描述你想分析的内容..."):
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.markdown(prompt)
        
        # AI响应
        with st.chat_message("assistant"):
            with st.spinner("思考中..."):
                response = app.chat(prompt)
            st.markdown(response)
        
        st.session_state.messages.append({"role": "assistant", "content": response})
"""

# ============ 部署检查清单 ============
DEPLOYMENT_CHECKLIST = """
LLM应用部署检查清单：

1. ✅ 错误处理
   - API重试机制（指数退避）
   - 超时处理
   - 降级策略（API不可用时的备选）

2. ✅ 监控
   - Token使用量统计
   - 响应时间追踪
   - 错误率监控

3. ✅ 安全
   - 输入验证和清洗
   - 输出内容审核
   - 防止Prompt注入

4. ✅ 成本控制
   - 设置每日/每月预算上限
   - 缓存常见查询结果
   - 使用更小的模型处理简单任务

5. ✅ 用户体验
   - 流式输出减少等待感
   - 加载状态提示
   - 错误信息友好化
"""


## 4. 练习题

1. **Chain构建**: 使用LangChain构建一个"新闻摘要→情感分析→生成配图描述"的Pipeline。
2. **文档问答**: 上传一篇你熟悉的领域的文章（如技术博客、学术论文），构建文档问答系统并测试至少10个问题。
3. **流式输出**: 实现一个带流式输出的聊天界面（使用Streamlit或Gradio），对比有/无流式输出时的用户体验差异。
4. **生产部署**: 将你构建的LLM应用部署到云平台（如Railway、Render、Vercel），配置监控和日志系统，验证生产环境的稳定性。
5. **完整项目**: 设计并实现一个完整的LLM应用（如私人助手、代码审查工具、内容生成平台），从需求分析、技术选型、代码实现、测试验证到部署上线全流程。

## 5. 延伸阅读

- **网站**: LangChain官方文档 — https://python.langchain.com/docs/
- **网站**: Streamlit文档 — https://docs.streamlit.io/
- **网站**: Gradio文档 — https://gradio.app/
- **GitHub**: LangChain Examples — https://github.com/gkamradt/llmtest
- **工具**: Weights & Biases — LLM应用监控 https://wandb.ai/
- **工具**: Helicone — LLM日志和监控 https://www.helicone.ai/

---

# 附录：课程进度检查清单

## L17 检查点
- [ ] 能解释GPT的自回归生成原理
- [ ] 掌握提示工程四种核心技巧
- [ ] 能分析LLM的局限性并提出缓解方案

## L18 检查点
- [ ] 能解释RAG的工作流程和优势
- [ ] 理解Embedding和向量数据库的核心概念
- [ ] 能实现完整的RAG Pipeline

## L19 检查点
- [ ] 能解释Agent的感知-规划-执行循环
- [ ] 掌握Tool Calling的实现方法
- [ ] 能使用LangChain构建Agent

## L20 检查点
- [ ] 掌握LangChain核心组件
- [ ] 能构建完整的LLM应用
- [ ] 了解生产环境部署的注意事项

---

*文档版本：v1.0 | 适用教材：AI 人工智能基础教程 Week 9-10*
